# Tutorial 8: Train NicheTrans on human lymph node data

In [31]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_human_lymph_node import Lymph_node

from utils.utils import *
from utils.utils_training_human_lymph_node import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [32]:
%run ./args/args_human_lymph_node.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.5, dropout_rate=0.1, use_moe_ffn=True, num_experts=4, moe_gate_hidden_dim=512, moe_gate_type='softmax', ffn_mult=2, n_source=3000, workers=4, adata_path='/mnt/datadisk0/Processed_DATA/2024_nm_human_lymph_nodes/', max_epoch=20, stepsize=10, train_batch=32, test_batch=32, optimizer='adam', lr=0.0003, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [33]:
# create the dataloaders
dataset = Lymph_node(adata_path=args.adata_path, n_top_genes=args.n_source)
trainloader, testloader = human_node_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.protein_length
model = NicheTrans(source_length=source_dimension, target_length=target_dimension, noise_rate=args.noise_rate, dropout_rate=args.dropout_rate,num_experts=args.num_experts,moe_gate_hidden_dim=args.moe_gate_hidden_dim)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

------Calculating spatial graph...
The graph contains 13638 edges, 3484 cells.
3.9145 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 27174 edges, 3484 cells.
7.7997 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 13138 edges, 3359 cells.
3.9113 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 26192 edges, 3359 cells.
7.7976 neighbors per cell on average.
=> Human lymph node loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  After filting  3484 spots
  test     |  After filting  3359 spots
  ------------------------------


### Initialize loss function (criterion) and optimizer

In [34]:
criterion = nn.MSELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [35]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader, device=device)
    if args.stepsize > 0: scheduler.step()
    ################
    
pearson = test(model, testloader, device=device)
torch.save(model.state_dict(), 'NicheTrans_human_lymph_node_last.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/20
Batch 108/108	 Loss 2.005607 (0.986351)
==> Epoch 2/20
Batch 108/108	 Loss 0.368745 (0.888354)
==> Epoch 3/20
Batch 108/108	 Loss 0.533794 (0.842017)
==> Epoch 4/20
Batch 108/108	 Loss 0.590651 (0.823233)
==> Epoch 5/20
Batch 108/108	 Loss 0.413441 (0.805568)
==> Epoch 6/20
Batch 108/108	 Loss 0.703781 (0.790263)
==> Epoch 7/20
Batch 108/108	 Loss 0.516164 (0.791156)
==> Epoch 8/20
Batch 108/108	 Loss 0.271678 (0.767474)
==> Epoch 9/20
Batch 108/108	 Loss 0.477673 (0.743450)
==> Epoch 10/20
Batch 108/108	 Loss 0.853367 (0.736896)
==> Epoch 11/20
Batch 108/108	 Loss 0.318920 (0.696699)
==> Epoch 12/20
Batch 108/108	 Loss 0.729135 (0.698600)
==> Epoch 13/20
Batch 108/108	 Loss 0.917123 (0.682186)
==> Epoch 14/20
Batch 108/108	 Loss 0.502593 (0.687694)
==> Epoch 15/20
Batch 108/108	 Loss 0.511639 (0.672992)
==> Epoch 16/20
Batch 108/108	 Loss 0.366591 (0.667766)
==> Epoch 17/20
Batch 108/108	 Loss 0.306220 (0.672407)
==> Epoch 18/20
Batch 108/108	 Loss 0.406718 (0.665175)
=